V1 (02 11 2025): Translated from R to Python and added the visualizations for goals B to D


V2 (04 01 2025): Loop system implemented

V2.5 (04 02 2025): Loops system tweaks. Removed openpyxl as it was corrupting files and replaced it with xlwings

In [15]:
#!pip install numpy pandas scipy matplotlib xlwings

In [16]:
import numpy as np
from scipy.stats import norm
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import os
import subprocess
import xlwings as xw
import time


# Define Functions
This section defines the functions used for calculating portfolio volatility, expected return, 
goal achievement probability, and the objective (failure probability) to minimize.

In [17]:

def sd_f(weight_vector, covar_table):
    covar_vector = np.zeros(len(weight_vector))
    for z in range(len(weight_vector)):
        covar_vector[z] = np.sum(weight_vector * covar_table[:, z])
    return np.sqrt(np.sum(weight_vector * covar_vector))

In [18]:
def mean_f(weight_vector, return_vector):
    return np.sum(weight_vector * return_vector)

In [19]:
def phi_f(goal_vector, goal_allocation, pool, mean, sd):
    # goal_vector is [value ratio, funding requirement, time horizon]
    required_return = (goal_vector[1] / (pool * goal_allocation))**(1 / goal_vector[2]) - 1
    if goal_allocation * pool >= goal_vector[1]:
        return 1
    else:
        return 1 - norm.cdf(required_return, loc=mean, scale=sd)

In [20]:
def optim_function(weights):
    # Uses the current global variables: goal_vector, allocation, pool, return_vector, covar_table
    return 1 - phi_f(
        goal_vector,
        allocation,
        pool,
        mean_f(weights, return_vector),
        sd_f(weights, covar_table)
    )

In [21]:
def constraint_function(weights):
    # For SciPy equality constraints, we require constraint_function(weights) == 0.
    return np.sum(weights) - 1

In [22]:
def mvu_f(weights):
    # mvu_f is defined for mean-variance optimization (not used below).
    return -(mean_f(weights, return_vector) - 0.5 * gamma * sd_f(weights, covariances)**2)

In [23]:
def r_req_f(goal_vector, goal_allocation, pool):
    return (goal_vector[1] / (goal_allocation * pool))**(1 / goal_vector[2]) - 1

# Load & Parse Data

In [24]:
# -- Variable Setup -- ##

#Monte Carlo Trials
n_trials = 10**5

#Case Study Profile Selection
Profile = "P1" #Either P1 or P2

#Excel worksheets
excel_returns = "Returns"
excel_volatilities = "Volatilities"
excel_correlation = "Correlation"
excel_gbi = "GBI Allocations P1" if Profile == "P1" else "GBI Allocations P2"
excel_gbi_goals = "GBI Goals P1" if Profile == "P1" else "GBI Goals P2"


In [25]:
## -- Repo Root and Folders -- ##

# Get repo root and set folders
root = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True).stdout.strip()
data_folder = os.path.join(root, "GBI Optimisation", "data")
output_folder = os.path.join(root, "GBI Optimisation")

# Get excel file and sheets
master_excel_path = os.path.join(data_folder, "Master.xlsx")

df_returns = pd.read_excel(master_excel_path, sheet_name=excel_returns)
df_vols = pd.read_excel(master_excel_path, sheet_name=excel_volatilities)

df_returns.set_index(df_returns.columns[0], inplace=True)
df_vols.set_index(df_vols.columns[0], inplace=True)

df_corr = pd.read_excel(master_excel_path, sheet_name=excel_correlation)
df_corr.set_index(df_corr.columns[0], inplace=True)


In [26]:
def get_goal_data(master_excel_path, plan=Profile):
    """
    Returns a DataFrame of the specified goal table (P1 or P2).
    P1 => B2:F5
    P2 => B7:F10
    """
    if plan == "P1":
        skip = 1  # start reading at row 2
    elif plan == "P2":
        skip = 6  # start reading at row 7
    else:
        raise ValueError("Plan not recognized. Use 'P1' or 'P2'.")

    df_goals = pd.read_excel(master_excel_path,sheet_name="Goals",skiprows=skip,nrows=4,usecols="B:F",header=0)
    # First column is "Goal Info", so make that the index
    df_goals.set_index(df_goals.columns[0], inplace=True)
    return df_goals

In [27]:
while True:
    # Reload loop table to get latest status
    table_loop_df = pd.read_excel(master_excel_path, sheet_name="Loop", usecols="B:F", skiprows=1, header=0)
    pending_rows = table_loop_df[table_loop_df["LoopStatus"] == "N"]

    if pending_rows.empty:
        print("All loops completed.")
        break

    # Get the next row to process
    chosen_row = pending_rows.loc[pending_rows["Year"].idxmin()]
    loop_year = chosen_row["Year"]
    loop_number = chosen_row["N"]
    loop_age1 = chosen_row["AgeP1"]
    loop_age2 = chosen_row["AgeP2"]

    print(f"Running optimization for year {loop_year}")

    # --- Begin your main processing block here ---

    # Reload wealth info
    df_wealth = pd.read_excel(master_excel_path, sheet_name="Wealth")
    df_wealth.set_index(df_wealth.columns[0], inplace=True)
    pool = df_wealth.loc[Profile, str(loop_year)]

    # Build capital market expectations
    capital_market_expectations_raw = {}
    for asset in df_returns.index:
        expected_return = df_returns.loc[asset, str(loop_year)]
        volatility = df_vols.loc[asset, 'volatility']
        capital_market_expectations_raw[asset] = {
            'Return Forecast': expected_return,
            'Volatility Forecast': volatility
        }
    capital_market_expectations_raw = pd.DataFrame.from_dict(capital_market_expectations_raw, orient='index')
    capital_market_expectations_raw = capital_market_expectations_raw.reset_index()
    capital_market_expectations_raw.rename(columns={'index': 'Unnamed: 0'}, inplace=True)

    # Get and adjust goals
    goal_data_raw = get_goal_data(master_excel_path, plan=Profile)
    starting_year = table_loop_df["Year"].min()
    goal_horizons = goal_data_raw.loc["Time Horizon"].astype(int)
    goal_years = starting_year + goal_horizons
    adjusted_horizons = goal_years - loop_year
    goal_data_raw.loc["Time Horizon"] = adjusted_horizons

    # Prepare data
    num_assets = capital_market_expectations_raw.shape[0]
    num_goals = goal_data_raw.shape[1]
    return_vector = capital_market_expectations_raw["Return Forecast"].to_numpy()
    correlations = df_corr.iloc[:num_assets, :num_assets].astype(float)
    stdevs = capital_market_expectations_raw["Volatility Forecast"].to_numpy()
    covariances = np.zeros((num_assets, num_assets))
    for i in range(num_assets):
        for j in range(num_assets):
            covariances[i, j] = stdevs[i] * stdevs[j] * correlations.iloc[i, j]

    # Parse goals
    goal_A = goal_data_raw["GOAL A"].values
    goal_B = goal_data_raw["GOAL B"].values
    goal_C = goal_data_raw["GOAL C"].values
    goal_D = goal_data_raw["GOAL D"].values

    # Optimization step 1: within-goal
    goal_allocation = np.arange(0.01, 1.01, 0.01)
    starting_weights = np.random.uniform(0, 1, num_assets)
    starting_weights /= np.sum(starting_weights)

    optimal_weights_A = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_B = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_C = np.zeros((len(goal_allocation), num_assets))
    optimal_weights_D = np.zeros((len(goal_allocation), num_assets))

    slsqp_opts = {'ftol': 1e-12, 'eps': 1e-12, 'maxiter': 10000, 'disp': False}

    for i, alloc in enumerate(goal_allocation):
        allocation = alloc
        covar_table = covariances

        for goal_var, goal_weights in zip([goal_A, goal_B, goal_C, goal_D],
                                           [optimal_weights_A, optimal_weights_B, optimal_weights_C, optimal_weights_D]):
            goal_vector = goal_var
            if goal_vector[1] <= pool * allocation:
                goal_weights[i, :] = [0]*(num_assets - 1) + [1]
            else:
                result = minimize(optim_function, starting_weights,
                                  constraints=[{'type': 'eq', 'fun': constraint_function}],
                                  bounds=[(0, 1)] * num_assets,
                                  method='SLSQP',
                                  options=slsqp_opts)
                goal_weights[i, :] = result.x

    # Step 2: Evaluate across-goal probabilities
    phi_A = np.zeros(len(goal_allocation))
    phi_B = np.zeros(len(goal_allocation))
    phi_C = np.zeros(len(goal_allocation))
    phi_D = np.zeros(len(goal_allocation))

    for i, alloc in enumerate(goal_allocation):
        phi_A[i] = phi_f(goal_A, alloc, pool,
                         mean_f(optimal_weights_A[i, :], return_vector),
                         sd_f(optimal_weights_A[i, :], covariances))
        phi_B[i] = phi_f(goal_B, alloc, pool,
                         mean_f(optimal_weights_B[i, :], return_vector),
                         sd_f(optimal_weights_B[i, :], covariances))
        phi_C[i] = phi_f(goal_C, alloc, pool,
                         mean_f(optimal_weights_C[i, :], return_vector),
                         sd_f(optimal_weights_C[i, :], covariances))
        phi_D[i] = phi_f(goal_D, alloc, pool,
                         mean_f(optimal_weights_D[i, :], return_vector),
                         sd_f(optimal_weights_D[i, :], covariances))

    # Simulate goal weights
    sim_goal_weights = np.random.multinomial(100, [1/num_goals]*num_goals, size=n_trials)
    for i in range(n_trials):
        rand_vector = np.random.uniform(0, 1, num_goals)
        percents = np.round((rand_vector / rand_vector.sum()) * 100, 0)
        sim_goal_weights[i, :] = np.maximum(percents, 1)

    utility = (
        goal_A[0] * phi_A[sim_goal_weights[:, 0] - 1] +
        goal_A[0] * goal_B[0] * phi_B[sim_goal_weights[:, 1] - 1] +
        goal_A[0] * goal_B[0] * goal_C[0] * phi_C[sim_goal_weights[:, 2] - 1] +
        goal_A[0] * goal_B[0] * goal_C[0] * goal_D[0] * phi_D[sim_goal_weights[:, 3] - 1]
    )

    index = np.argmax(utility)
    optimal_goal_weights = sim_goal_weights[index, :]

    optimal_subportfolios = np.zeros((num_goals, num_assets))
    goals = ["A", "B", "C", "D"]
    for i in range(num_goals):
        optimal_subportfolios[i, :] = eval(f"optimal_weights_{goals[i]}")[optimal_goal_weights[i] - 1, :]

    optimal_aggregate_portfolio = (optimal_goal_weights / 100) @ optimal_subportfolios
    asset_names = capital_market_expectations_raw.iloc[:, 0].astype(str).tolist()

    # Prepare Excel outputs
    raw_alloc = optimal_aggregate_portfolio * 100
    normalized_alloc = raw_alloc / raw_alloc.sum() * 100
    normalized_goal_alloc = optimal_goal_weights / np.sum(optimal_goal_weights) * 100

    df_across_goal = pd.DataFrame({
        "Goal": goals,
        "Allocation (%)": np.round(normalized_goal_alloc, 2)
    })

    df_aggregate = pd.DataFrame({
        "Asset": asset_names,
        "Weight": optimal_aggregate_portfolio,
        "Allocation (%)": np.round(normalized_alloc, 2)
    })

    # Write to Excel
    app = xw.App(visible=False)
    app.display_alerts = False
    app.screen_updating = False
    wb = app.books.open(master_excel_path)

    # Aggregate weights
    ws = wb.sheets[excel_gbi]
    header_values = [ws.cells(1, col).value for col in range(2, 52)]
    year_col = header_values.index(str(loop_year)) + 2
    for i, asset in enumerate(asset_names):
        allocation = float(np.round(normalized_alloc[i], 2)) / 100
        cell = ws.cells(i + 2, year_col)
        cell.value = allocation
        cell.number_format = '0.00%'

    # Goal weights
    ws_goals = wb.sheets[excel_gbi_goals]
    goal_header_values = [ws_goals.cells(1, col).value for col in range(2, 52)]
    goal_year_col = goal_header_values.index(str(loop_year)) + 2
    for i, allocation in enumerate(df_across_goal["Allocation (%)"]):
        value = float(np.round(allocation / 100, 6))
        row = i + 2
        cell = ws_goals.cells(row, goal_year_col)
        cell.value = value
        cell.number_format = '0.00%'

    # Update loop status
    ws_loop = wb.sheets["Loop"]
    last_row = ws_loop.cells.last_cell.row
    loop_data = ws_loop.range("B2:F" + str(last_row)).value

    for i, row in enumerate(loop_data):
        year = row[1]
        status = row[4]
        if year == loop_year and status == 'N':
            ws_loop.cells(i + 2, 6).value = 'Y'
            print(f"Updated loop status for year {loop_year}")
            break

    wb.save()
    wb.close()
    app.quit()
    time.sleep(1)


Running optimization for year 2042
Updated loop status for year 2042
Running optimization for year 2043


C:\Users\admin\AppData\Local\Temp\ipykernel_15248\4021808357.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  required_return = (goal_vector[1] / (pool * goal_allocation))**(1 / goal_vector[2]) - 1


Updated loop status for year 2043
Running optimization for year 2044
Updated loop status for year 2044
Running optimization for year 2045
Updated loop status for year 2045
Running optimization for year 2046
Updated loop status for year 2046
Running optimization for year 2047
Updated loop status for year 2047
Running optimization for year 2048
Updated loop status for year 2048
Running optimization for year 2049
Updated loop status for year 2049
Running optimization for year 2050
Updated loop status for year 2050
Running optimization for year 2051
Updated loop status for year 2051
Running optimization for year 2052
Updated loop status for year 2052
Running optimization for year 2053
Updated loop status for year 2053
Running optimization for year 2054
Updated loop status for year 2054
Running optimization for year 2055


C:\Users\admin\AppData\Local\Temp\ipykernel_15248\4021808357.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  required_return = (goal_vector[1] / (pool * goal_allocation))**(1 / goal_vector[2]) - 1


Updated loop status for year 2055
Running optimization for year 2056
Updated loop status for year 2056
Running optimization for year 2057
Updated loop status for year 2057
Running optimization for year 2058
Updated loop status for year 2058
Running optimization for year 2059
Updated loop status for year 2059
Running optimization for year 2060
Updated loop status for year 2060
Running optimization for year 2061
Updated loop status for year 2061
Running optimization for year 2062
Updated loop status for year 2062
Running optimization for year 2063
Updated loop status for year 2063
Running optimization for year 2064
Updated loop status for year 2064
Running optimization for year 2065
Updated loop status for year 2065
Running optimization for year 2066
Updated loop status for year 2066
Running optimization for year 2067
Updated loop status for year 2067
Running optimization for year 2068
Updated loop status for year 2068
Running optimization for year 2069
Updated loop status for year 2069


KeyError: '2075'

                             Unnamed: 0  Return Forecast  Volatility Forecast
0           Emerging Markets - Equities         0.040017                0.086
1          Developed Markets - Equities        -0.006662                0.070
2  Emerging Markets State - Obligations         0.079198                0.049
3        High Yield Bonds - Obligations         0.099826                0.050
4  Investment Grade Bonds - Obligations         0.095851                0.037
5     Government ZC Bonds - Obligations         0.083761                0.029


                      GOAL A      GOAL B    GOAL C      GOAL D
Goal Info                                                     
Value Ratio                1        0.45       0.5        0.58
Funding Requirement  5157000  5000000.00  713500.0  8812000.00
Time Horizon              10       30.00       4.0       18.00


Starting Year: 2025
Current Year: 2041
Goal Years: {'GOAL A': 2035, 'GOAL B': 2055, 'GOAL C': 2029, 'GOAL D': 2043}
Adjusted Horizons: {'GOAL A': -6, 'GOAL B': 14, 'GOAL C': -12, 'GOAL D': 2}


# Parse Goal Data
Each goal vector is of the form: [value ratio, funding requirement, time horizon]

                      GOAL A      GOAL B    GOAL C      GOAL D
Goal Info                                                     
Value Ratio                1        0.45       0.5        0.58
Funding Requirement  5157000  5000000.00  713500.0  8812000.00
Time Horizon              -6       14.00     -12.0        2.00


# Step 1: Optimal Within-Goal Allocation
Enumerate possible across-goal allocations (from 0.01 to 1)
and, for each goal, optimize the subportfolio weights.

# Step 2: Optimal Across-Goal Allocation
Simulate goal weights and compute utility for each trial.

# Step 3: Optimal Subportfolios & Aggregate Portfolio
Retrieve the optimal subportfolio allocations and compute the aggregate portfolio.

# Print Results

Optimal Across-Goal Allocation:
[ 2 96  1  1]

Optimal Aggregate Investment Allocation:
[0.00392107 0.00810492 0.00905063 0.96881346 0.00298443 0.00712549]


Optimal Across-Goal Allocation:
Goal  Allocation (%)
   A             2.0
   B            96.0
   C             1.0
   D             1.0

Optimal Aggregate Investment Allocation:
                               Asset   Weight  Allocation (%)
         Emerging Markets - Equities 0.003921            0.39
        Developed Markets - Equities 0.008105            0.81
Emerging Markets State - Obligations 0.009051            0.91
      High Yield Bonds - Obligations 0.968813           96.88
Investment Grade Bonds - Obligations 0.002984            0.30
   Government ZC Bonds - Obligations 0.007125            0.71


Starting loop status check...
Accessed 'Loop' worksheet.
Last cell row: 1048576
Loaded loop data. Total rows read: 1048575
Row 2: Year = Year, Status = LoopStatus
Row 3: Year = 2025.0, Status = Y
Row 4: Year = 2026.0, Status = Y
Row 5: Year = 2027.0, Status = Y
Row 6: Year = 2028.0, Status = Y
Row 7: Year = 2029.0, Status = Y
Row 8: Year = 2030.0, Status = Y
Row 9: Year = 2031.0, Status = Y
Row 10: Year = 2032.0, Status = Y
Row 11: Year = 2033.0, Status = Y
Row 12: Year = 2034.0, Status = Y
Row 13: Year = 2035.0, Status = Y
Row 14: Year = 2036.0, Status = Y
Row 15: Year = 2037.0, Status = Y
Row 16: Year = 2038.0, Status = Y
Row 17: Year = 2039.0, Status = Y
Row 18: Year = 2040.0, Status = Y
Row 19: Year = 2041.0, Status = N
Match found at row 19. Updating status to 'Y'.
Status updated successfully.
